# fiftyone_preview.ipynb — preview a class BEFORE pulling it

**When to use this:** *before* running `acquire_openimages.py` for a class (or re-running it after a filter change). Pulls that class at the same real, buffered volume the real script would use, straight from the Open Images Zoo — network required. Nothing is saved to disk; changing `class_key` and re-running overwrites the same throwaway FiftyOne dataset.

**Why it exists:** to catch filter/volume problems (wrong native class name, too few/many results, depiction or group-of boxes you don't want) *before* spending a real multi-GB pull on them.

**Not for:** browsing data that's already on disk — use `fiftyone_explore.ipynb` (raw) or `fiftyone_review_processed.ipynb` (converted) for that instead.

In [ ]:
# Imports -- reuses acquire_openimages.py's config-driven lookups
# (native class names, cap, buffer math) instead of hand-typing them,
# so a typo here can't silently pull the wrong/unfiltered class the
# way it did in fiftyone_test.ipynb ("Waste Container" vs. the real
# "Waste container").
import math
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk up from `start` to find the repo root (has config/ + AGENTS.md).

    Needed because notebooks live in notebooks/, not the repo root, and
    Jupyter's working directory depends on how it was launched — this
    makes the scripts.* import below robust regardless of that.
    """
    for parent in [start, *start.parents]:
        if (parent / "config").is_dir() and (parent / "AGENTS.md").is_file():
            return parent
    raise RuntimeError("Could not locate repo root from notebook cwd.")


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

from scripts.acquire.acquire_openimages import (
    get_openimages_targets,
    compute_max_samples,
    LABEL_FIELD,
    ZOO_DATASET_NAME,
    ZOO_SPLITS,
    SEED,
)

targets = get_openimages_targets()
PREVIEW_DATASET_NAME = "preview_openimages"

In [ ]:
# Change this and re-run cells 3-5 to preview a different class.
# Each run overwrites the same preview dataset -- nothing is exported
# or saved to disk here, this is look-before-you-pull only.
class_key = "chairs"

In [ ]:
# Pull directly from the Zoo at real (buffered) volume, same filtering
# acquire_openimages.py applies (DEC-043) -- tweak the filter expression
# here freely to experiment before touching the real script.
info = targets[class_key]
max_samples = compute_max_samples(info["cap"])
# max_samples applies PER split when `splits` (plural) is passed, not as a
# combined total (verified directly, same fix as DEC-044) -- divide across
# the three splits so the preview matches what the real script would pull.
per_split_max_samples = math.ceil(max_samples / len(ZOO_SPLITS))
print(f"{class_key}: native={info['native_classes']} max_samples={max_samples} (~{per_split_max_samples}/split)")

dataset = foz.load_zoo_dataset(
    ZOO_DATASET_NAME,
    splits=list(ZOO_SPLITS),
    label_types=["detections"],
    classes=info["native_classes"],
    max_samples=per_split_max_samples,
    shuffle=True,
    seed=SEED,
    dataset_name=PREVIEW_DATASET_NAME,
    drop_existing_dataset=True,
    label_field=LABEL_FIELD,
)

view = dataset.filter_labels(
    LABEL_FIELD,
    (F("IsDepiction") == False) & (F("IsGroupOf") == False),
)
view = view.match(F(f"{LABEL_FIELD}.detections").length() > 0)
print(f"{len(dataset)} pulled -> {len(view)} after filtering")

In [ ]:
# Launch the App
session = fo.launch_app(view, auto=False)

In [ ]:
session